# ODS v2: GA4 web sessions

Builds 1 ods tables in `ods_v2`: `WebSession`.

Depends on no other build notebook.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the description above the failing cell: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("31_Web_Session", "started")


## WebSession


**`${ods.stg}.odsv2_ga4_ref_sessions`**

GA4 client sessions with a Motor or Home quote reference on any page view.

- **Reads:** dlk.ga4_page_views_daily

In [ ]:
create or replace table ${ods.stg}.odsv2_ga4_ref_sessions as
select  distinct ClientID, SessionID
from    ${ods.dlk}.ga4_page_views_daily
where   EventDate >= date '${ods.cutoff_date}'
and     (nullif(trim(MotorQuoteReference), '') is not null or nullif(trim(HomeQuoteReference), '') is not null)

**`{target}.WebSession`**

Every GA4 session, quote or not, with what it looked at: page views, views of content pages (Chill's PagePathGroup Chill Blog, Chill Cust Care, Chill Company, Insurers) and one flag per product page group. HasQuoteReference marks a session with a Motor or Home quote reference (dlk page views) or an MFQ quote guid. EmailKey is the salted hash of the email the warehouse ties to the GA cookie from its MFQ quotes (CustomerReference under "Rule 1: ClientID 1:1 EMail" or "Rule 2: Multiple EMails"); the email itself is not held. SessionID is unique only with ClientID.

- **Grain:** one row per GA4 session (client and session id)
- **Business key:** ClientID + SessionID
- **History:** event
- **Cut-off:** SessionDate
- **Source:** edw.tbl_fact_ga_clicks_ga4 (+ dlk.ga4_page_views_daily quote references)
- **Reads:** edw.tbl_fact_ga_clicks_ga4, stg.odsv2_ga4_ref_sessions

In [ ]:
create or replace table ${ods.target}.WebSession as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.ClientID,
                c.SessionID,
                min(c.WorkDate)                                                 as SessionDate,
                min(c.WorkDateTime)                                             as SessionStartUtc,
                count(*)                                                        as PageViews,
                sum(case when c.PagePathGroup in ('Chill Blog', 'Chill Cust Care', 'Chill Company', 'Insurers') then 1 else 0 end)
                                                                                as ContentViews,
                max(case when c.PagePathGroup = 'Motor' then 1 else 0 end)      as ViewedMotor,
                max(case when c.PagePathGroup = 'Home' then 1 else 0 end)       as ViewedHome,
                max(case when c.PagePathGroup = 'Van' then 1 else 0 end)        as ViewedVan,
                max(case when c.PagePathGroup = 'Travel' then 1 else 0 end)     as ViewedTravel,
                max(case when c.PagePathGroup = 'Life' then 1 else 0 end)       as ViewedLife,
                max(case when c.PagePathGroup = 'Gadget' then 1 else 0 end)     as ViewedGadget,
                max(case when c.PagePathGroup = 'Commercial' then 1 else 0 end) as ViewedCommercial,
                max(case when c.PagePathGroup = 'Renewals' then 1 else 0 end)   as ViewedRenewals,
                max(case when r.SessionID is not null or nullif(trim(c.`MFQ_QuoteQueryGUID1`), '') is not null then 1 else 0 end)
                                                                                as HasQuoteReference,
                max(e.EmailKey)                                                 as EmailKey,
                count(distinct e.EmailKey)                                      as LinkedContactCount
        from    ${ods.edw}.tbl_fact_ga_clicks_ga4 c
        left join ${ods.stg}.odsv2_ga4_ref_sessions r on r.ClientID = c.ClientID and r.SessionID = c.SessionID
        left join (select  CustomerReference, ods_hash_email(CustomerReference) as EmailKey
                   from    (select distinct CustomerReference
                            from   ${ods.edw}.tbl_fact_ga_clicks_ga4
                            where  WorkDate >= date '${ods.cutoff_date}'
                            and    (CustomerReferenceRule like 'Rule 1%' or CustomerReferenceRule like 'Rule 2%')) d
                   ) e
                on  e.CustomerReference = c.CustomerReference
                and (c.CustomerReferenceRule like 'Rule 1%' or c.CustomerReferenceRule like 'Rule 2%')
        where   c.WorkDate >= date '${ods.cutoff_date}'
        and     nullif(trim(c.SessionID), '') is not null
        and     nullif(trim(c.ClientID), '') is not null
        group by c.ClientID, c.SessionID
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "WebSession", "bk": ["ClientID", "SessionID"], "sk": null, "nullable": [], "allow": [], "latest": null}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    if s["latest"]:                                     # exactly one latest row per group (e.g. per policy)
        lf, per = s["latest"]["column"], s["latest"]["per"]
        bad = (df.groupBy(*per).agg(F.sum(F.col(lf)).alias("k")).where("k is null or k <> 1").count())
        if bad:
            problems.append(f"{bad:,} groups without exactly one {lf} row")
            advice.append(f"{lf} must be 1 on exactly one row per {' + '.join(per)}: the window in the cell no longer "
                          f"matches the business key, or the source sent two rows for the latest version")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("31_Web_Session", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("31_Web_Session", "succeeded")
